## Project Objective

The aim of the project is to identify the revenue structure and segment customers based on the RFM (Recency, Frequency, Monetary) model in order to answer key business questions:

- How concentrated is revenue?
- Which customers generate the most value?
- Are there distinct segments with different purchasing behaviors?
- What are the risks of dependence on a narrow group of customers or products?

The project is decision-making in nature—its task is not only to describe the data, but also to identify areas requiring priority action (retention, upsell, concentration risk management).

The end result is a structured customer value map that enables:

- marketing strategy optimization,
- better customer relationship management,
- identification of high- and low-risk segments.

### 1] Importing all the libraries necessary for this project

In [6]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os
from sqlalchemy import create_engine
from sqlalchemy import text

sys.path.insert(0, os.path.abspath('..'))
from src.config import get_engine

### 2] Load and Overview Data from CSV

In [ ]:
engine = get_engine()

In [ ]:
df = pd.read_sql("SELECT * FROM transactions", engine)

In [ ]:
# Head of the dataset
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
# Check null values
df.isna().sum()

In [ ]:
# Negative price - check if double negative values
df[(df['UnitPrice']<0) & (df['Quantity']<0)]

In [ ]:
# No double negatives - check if negative price is incorrect value
df[df['UnitPrice']<0]

Description "Adjust bad debt" -> not related to RFM

### 3] Data cleaning

In [ ]:
with open("../sql/create_clean_view.sql", "r") as file:
    query = file.read()

with engine.connect() as conn:
    conn.execute(text(query))
    conn.commit()

In [ ]:
df_clean = pd.read_sql(
    "SELECT * FROM clean_transactions;",
    engine
)

df_clean.head()

In [ ]:
# Fix types
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])
df_clean['Country'] = df_clean['Country'].astype('category')
df_clean.describe()

### 4] Data analysis

#### A) Customer Revenue & RFM Analysis
##### 1. Data Overview

The dataset contains 541,909 transactional records with the following variables:

- Invoice information

- Product information

- Quantity and Unit Price

- Customer ID

- Country

After removing:

- Transactions with non-positive Quantity

- Transactions with non-positive UnitPrice

- Rows without CustomerID

the final analytical dataset contains **397,884 valid purchase records**.

The cleaning process ensures that only completed, revenue-generating transactions are included in the customer-level analysis.

**Create customer table**

Start with basic aggregation. Values per customer

In [ ]:
df_full = df_clean.copy()

In [ ]:
customer_table = df_full.groupby('CustomerID').agg(
    TotalRevenue=('Revenue', 'sum'),
    TotalOrders=('InvoiceNo', 'nunique'),
    TotalTransactions=('InvoiceNo', 'count'),
    TotalQuantity=('Quantity', 'sum'),
    FirstPurchase=('InvoiceDate', 'min'),
    LastPurchase=('InvoiceDate', 'max')
).reset_index()
customer_table.head()

In [ ]:
customer_table['AvgOrderValue'] = (
    customer_table['TotalRevenue'] / customer_table['TotalOrders']
)
customer_table.head()

In [ ]:
snapshot_date = df_clean['InvoiceDate'].max() + pd.Timedelta(days=1)

In [ ]:
customer_table['Recency'] = (
    snapshot_date - customer_table['LastPurchase']
).dt.days
customer_table.head()

#### B) Customer-Level Aggregation

Transactional data was aggregated to the customer level.
For each customer, the following metrics were calculated:

- **TotalRevenue** – total monetary value generated

- **TotalOrders** – number of unique invoices

- **TotalTransactions** – number of rows (line items)

- **TotalQuantity** – total number of purchased items

- **AvgOrderValue** – revenue per order

- **Recency** – number of days since last purchase

This transformation shifts the analysis from transaction-level behavior to customer-level behavior.

Total number of customers: **4,338**

#### C) Pareto analysis

In [ ]:
pareto = pd.read_sql("""
WITH revenue_ranked AS (
    SELECT
        "CustomerID",
        SUM("Revenue") AS total_revenue
    FROM clean_transactions
    GROUP BY "CustomerID"
),
ordered AS (
    SELECT
        "CustomerID",
        total_revenue,
        RANK() OVER (ORDER BY total_revenue DESC) AS revenue_rank,
        total_revenue / SUM(total_revenue) OVER () AS revenue_share,
        SUM(total_revenue) OVER (ORDER BY total_revenue DESC)
            / SUM(total_revenue) OVER () AS cumulative_share,
        COUNT(*) OVER () AS total_customers
    FROM revenue_ranked
)
SELECT *,
       revenue_rank::float / total_customers AS customer_percentile
FROM ordered
ORDER BY revenue_rank;
""", engine)
pareto.head()

In [ ]:
pareto_80 = pareto[pareto['cumulative_share'] <= 0.8]
pareto_ratio = len(pareto_80) / pareto['CustomerID'].nunique()
pareto_ratio

In [ ]:
plt.plot(pareto['customer_percentile'], pareto['cumulative_share'])
plt.axhline(0.8, linestyle='--')
plt.axvline(pareto_ratio, linestyle='--')
plt.gca().yaxis.set_major_formatter(lambda x, _: f'{x:.0%}')
plt.grid(alpha=0.3)
plt.title("Pareto Curve")
plt.xlabel("Customer Percentile")
plt.ylabel("Cumulative Revenue Share")
plt.show()

The distribution of customer revenue is highly right-skewed.
The majority of customers generate relatively low revenue and place a small number of orders, while a small subset of customers accounts for a disproportionately large share of total revenue.
This long-tail structure is typical for retail and e-commerce businesses.

In [ ]:
# Revenue distribution
pareto['total_revenue'].describe()

In [ ]:
#bar diagram

plt.hist(pareto['total_revenue'], bins=50)
plt.title("Distribution of Customer Revenue")
plt.show()

#### D) Revenue Distribution & Pareto Analysis

Customers were sorted by TotalRevenue, and cumulative revenue share was calculated.

**Result:**

Approximately 26% of customers generate 80% of total revenue.

This indicates:

- Revenue is concentrated

- But not extremely concentrated (not 5–10%)

- A significant upper quartile drives business performance

This is a typical long-tail distribution observed in retail and e-commerce environments.

The histogram confirms strong right-skewness:

- Majority of customers generate relatively low revenue

- Small group of customers generate disproportionately high revenue

Business implication:
Retention of high-value customers is strategically critical.

**RFM Analysis**

In [ ]:
# sort customers by revenue
customer_table = customer_table.sort_values('TotalRevenue', ascending=False)
customer_table.head()

In [ ]:
customer_table['R_Score'] = pd.qcut(
    customer_table['Recency'],
    q=4,
    labels=[4,3,2,1],
    duplicates='drop'
)
customer_table['F_Score'] = pd.qcut(customer_table['TotalOrders'].rank(method='first'), 4, labels=[1,2,3,4])
customer_table['M_Score'] = pd.qcut(customer_table['TotalRevenue'], 4, labels=[1,2,3,4])

customer_table['RFM_Score'] = (
    customer_table['R_Score'].astype(str) +
    customer_table['F_Score'].astype(str) +
    customer_table['M_Score'].astype(str)
)

In [ ]:
def segment(row):
    if row['R_Score']==4 and row['F_Score']>=3 and row['M_Score']>=3:
        return "Champions"
    elif row['R_Score']<=2 and row['F_Score']<=2:
        return "At Risk"
    else:
        return "Regular"

customer_table['Segment'] = customer_table.apply(segment, axis=1)

customer_table.head()

#### E) RFM Analysis

RFM segmentation was performed using quartiles:

- **R (Recency)** – how recently a customer purchased

- **F (Frequency)** – number of unique orders

- **M (Monetary)** – total revenue

Each dimension was divided into 4 quantiles.

This allows segmentation such as:

- High R, High F, High M → Core customers

- Low R, Low F, Low M → At-risk or low-value customers

The analysis confirms a clear stratification within the customer base, enabling targeted marketing or retention strategies.

In [ ]:
rfm_revenue = (
    customer_table
    .pivot_table(
        index='R_Score',
        columns='F_Score',
        values='TotalRevenue',
        aggfunc='mean',
        observed=False
    )
)

In [ ]:
plt.figure(figsize=(8,6))
plt.imshow(rfm_revenue, aspect='auto')

plt.colorbar(label='Average Revenue')

plt.xticks(
    ticks=np.arange(len(rfm_revenue.columns)),
    labels=rfm_revenue.columns
)

plt.yticks(
    ticks=np.arange(len(rfm_revenue.index)),
    labels=rfm_revenue.index
)

plt.xlabel("Frequency Score")
plt.title("Average Revenue by RFM Segment")
plt.grid(False)

plt.tight_layout()
plt.show()

#### F) Revenue by RFM Segment

The chart shows the average revenue generated by customers broken down into RFM segments (e.g., R4F4M4, R1F1M1, etc.).

Each segment represents a combination of:

**- Recency**

**- Frequency** 

**- Monetary**


**Analytical conclusions**

**1. The highest average revenue is generated by segments with high M (Monetary)**
    Monetary is the strongest determinant of average revenue per customer, which is logical, but confirms the correctness of the segmentation.

**2. Segments with high R and F further strengthen the value of M**
    Customers who buy frequently and recently maintain a high level of revenue generated — this is the most stable revenue group.

**3. Segments with low R (have not bought for a long time) have lower average revenue**
    This may indicate a fading relationship with the brand or the customer's life cycle.

**4. Clear heterogeneity between segments**
    Average revenue varies significantly between RFM combinations, which means that segmentation effectively separates different customer economic profiles.


**Business interpretation**

- R4F4M4 segments → strategic customers (core revenue drivers)

- Segments with high M but low R → potential risk of losing a valuable customer

- Segments with high F, low M → candidates for upselling

- Segments low in all dimensions → low value, low priority


**Strategic conclusion**

RFM segmentation effectively differentiates customers in terms of economic value.
The Monetary component is most important for revenue, while Recency acts as a risk indicator.

In [ ]:
snapshot_date = df_full['InvoiceDate'].max() + pd.Timedelta(days=1)

# Assume that 'lost' means no purchase in the last year
lost_customers = customer_table[customer_table['LastPurchase'] < (snapshot_date - pd.Timedelta(days=365))]

len(lost_customers)

#### G) Lost Customers

Definition used:
Customers with no purchase within the last 365 days relative to dataset snapshot.

Result:
77 customers classified as lost

Given total customer base (4,338), churn level appears relatively low.

In [ ]:
# segment wg M_Score (Monetary)
revenue_by_segment = customer_table.groupby('M_Score', observed=True)['TotalRevenue'].sum().sort_values(ascending=False)
revenue_by_segment

#### H) Revenue by Segment (Monetary)

Revenue distribution by Monetary score:

- Segment 4 (highest spenders) generates dominant share of revenue

- Clear gradient from M4 → M1

This confirms monetary segmentation is meaningful and reflects actual business value differences.

In [ ]:
segment_stats = (
    customer_table
    .groupby('M_Score', observed=True)
    .agg(
        Customers=('CustomerID','count'),
        TotalRevenue=('TotalRevenue','sum'),
        AvgRevenue=('TotalRevenue','mean')
    )
)

segment_stats['RevenueShare'] = (
    segment_stats['TotalRevenue'] /
    segment_stats['TotalRevenue'].sum()
)

segment_stats

In [ ]:
plt.figure(figsize=(8,5))
plt.bar(revenue_by_segment.index.astype(str),
        revenue_by_segment.values)

plt.title("Revenue by Monetary Segment")
plt.xlabel("Monetary Score (M)")
plt.ylabel("Total Revenue")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

#### I) Revenue by Monetary Segment

The chart shows the total revenue generated by customers broken down into Monetary segments (M1–M4), where:

**M4** → highest basket value / highest total revenue per customer

**M1** → lowest value

Segmentation is based on quartiles.


**Analytical conclusions**

**1. Strong concentration of revenue in the M4 segment**
     The highest segment generates the dominant part of total revenue.
     This confirms the classic long-tail distribution.

**2. Clear separation between segments**
     The revenue gradient M4 → M1 is clear, which means that the Monetary score effectively distinguishes customers in terms of business value.

**3. The M1 segment has a marginal impact on financial performance**
     A large number of low-spending customers does not translate proportionally into revenue.

**Business implications**

- Retention of the M4 segment should be an absolute priority.
- Losing even a small percentage of M4 customers can have a significant impact on financial performance.
- The M2–M3 segment may be a potential target for upselling campaigns.
- The M1 segment requires low-cost, automated marketing activities — it does not justify high CAC.

**Strategic conclusion**

The business model is heavily dependent on top-tier customers.
This increases efficiency, but also raises the risk of revenue concentration.

In [ ]:
revenue_by_country = df_full.groupby('Country', observed=False)['Revenue'].sum().sort_values(ascending=False)
revenue_by_country.head()

In [ ]:
uk_share = revenue_by_country['United Kingdom'] / revenue_by_country.sum()
uk_share

#### J) Revenue by Country

Revenue concentration by geography:

- United Kingdom accounts for approximately 82% of total revenue

This indicates:

- Strong domestic market dependence

- Limited international diversification

Business risk:
High exposure to single market.

In [ ]:
df_full['YearMonth'] = df_full['InvoiceDate'].dt.to_period('M')
monthly_revenue = df_full.groupby('YearMonth')['Revenue'].sum()
monthly_revenue.plot(kind='line', figsize=(12,5), title='Monthly Revenue')

#### K) Monthly Seasonality

Monthly revenue analysis shows:

- Clear growth trend throughout 2011

- Strong peak in Q4

- Seasonal acceleration toward year-end

This pattern is consistent with retail seasonality (holiday effect).

In [ ]:
top10_products = pd.read_sql("""SELECT "Description",
       SUM("Quantity" * "UnitPrice") AS "Revenue"
FROM clean_transactions
GROUP BY "Description"
ORDER BY "Revenue" DESC
LIMIT 10;""", engine)
top10_products

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(top10_products["Description"], top10_products['Revenue'])
plt.gca().invert_yaxis()

plt.xlabel("Revenue")
plt.title("Top 10 Products by Revenue")
plt.tight_layout()
plt.show()

#### L) Top 10 Products by Revenue

The chart shows the 10 products that generate the highest total revenue (Quantity × UnitPrice).

**Analytical conclusions**

**1. Visible concentration of revenue on a limited number of products.**
     The top 10 products account for a significant share of total revenue.

**2. Pareto effect at the product level**
     A small percentage of SKUs generates a significant portion of sales.

**3. Possible high sensitivity of the business to the availability of these products**
     Logistical problems, stock shortages, or price changes can significantly affect financial results.

**Business implications**

- Top 10 products should be prioritized in:
    - inventory management
    - promotions
    - margin analysis
- negotiations with suppliers

Bundling or cross-selling around these SKUs can be considered.

**Strategic conclusion**

The business shows not only customer concentration but also product concentration.
This increases operational efficiency but at the same time raises the risk of dependence on a narrow portfolio.

### 5] Final Business Insights
- Revenue concentration risk
- High dependency on domestic market
- Strong seasonality Q4
- Clear segmentation potential
- High-value customers require retention strategy